# Market-Making Strategy Comparison

Three strategies on identical synthetic order flow:
1. **Naive** — symmetric ±k tick quotes, fixed size
2. **Avellaneda-Stoikov** — optimal reservation price + spread derived from CARA utility
3. **Glosten-Milgrom** — flow-imbalance-aware spread widening

All use the same C++ matching engine (price-time priority LOB) and the same Hawkes-process order flow.

In [ ]:
import sys, os
# Make sure the project root and build dir are on path.
PROJECT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BUILD   = os.path.join(PROJECT, 'build')
for p in (PROJECT, BUILD):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from data.synthetic import generate_hawkes
from sim.backtester import Backtester
from sim.metrics import compute_metrics
from strategies.naive import NaiveMarketMaker
from strategies.avellaneda_stoikov import AvellanedaStoikov
from strategies.glosten_milgrom import GlostenMilgrom

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

## 1. Generate synthetic order flow

Hawkes process: self-exciting arrivals that cluster in time (mimics real order bursts).

In [ ]:
FLOW = generate_hawkes(
    T_seconds=60.0,
    baseline_rate=300.0,
    alpha=0.6,
    beta=1.0,
    mid_price=10_000,
    tick_std=15.0,
    mean_qty=8.0,
    market_fraction=0.20,
    seed=42,
)
print(f'Generated {len(FLOW):,} orders over 60 s')

# Quick look at inter-arrival distribution
ts = np.array([o.ts_us for o in FLOW])
iats = np.diff(ts) / 1e3  # milliseconds

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(iats, bins=80, color='steelblue', edgecolor='none')
ax.set_xlabel('Inter-arrival time (ms)')
ax.set_ylabel('Count')
ax.set_title('Hawkes-process inter-arrival distribution')
ax.set_xlim(0, np.percentile(iats, 99))
plt.tight_layout()
plt.savefig('inter_arrival.png', dpi=120)
plt.show()

## 2. Run all three strategies

In [ ]:
T_HORIZON = 60.0  # seconds — must match Hawkes simulation

strategies = {
    'Naive':              NaiveMarketMaker(half_spread_ticks=4, order_size=5, max_inventory=40),
    'Avellaneda-Stoikov': AvellanedaStoikov(gamma=0.15, sigma=2.5, k=1.5,
                                            T=T_HORIZON, order_size=5, max_inventory=40),
    'Glosten-Milgrom':    GlostenMilgrom(base_half_spread=4, lambda_factor=2.5,
                                          skew_factor=0.4, window=40, order_size=5, max_inventory=40),
}

results = {}
for name, strat in strategies.items():
    bt = Backtester(strategy=strat, tick_size=0.01, initial_cash=0.0)
    bt.run(FLOW)
    results[name] = (bt, compute_metrics(bt))
    m = results[name][1]
    print(f'{name:22s}  PnL={m.total_pnl:+.2f}  Sharpe={m.sharpe:+.2f}  '
          f'Fills={m.fill_count:4d}  MaxDD={m.max_drawdown:.2f}')

## 3. Metrics summary table

In [ ]:
rows = {name: m.to_dict() for name, (_, m) in results.items()}
df = pd.DataFrame(rows).T
df.index.name = 'Strategy'
df

## 4. Side-by-side plots

In [ ]:
COLORS = {'Naive': '#2196F3', 'Avellaneda-Stoikov': '#4CAF50', 'Glosten-Milgrom': '#FF5722'}

fig = plt.figure(figsize=(15, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Row 0: Cumulative PnL
axes_pnl = [fig.add_subplot(gs[0, i]) for i in range(3)]
# Row 1: Inventory over time
axes_inv = [fig.add_subplot(gs[1, i]) for i in range(3)]
# Row 2: Inventory distribution
axes_hist = [fig.add_subplot(gs[2, i]) for i in range(3)]

for col, (name, (bt, m)) in enumerate(results.items()):
    color = COLORS[name]
    t_s   = m.ts_us / 1e6  # convert to seconds

    # ── PnL ──
    ax = axes_pnl[col]
    ax.plot(t_s, m.pnl, color=color, lw=1.2)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Time (s)')
    if col == 0: ax.set_ylabel('PnL ($)')

    # ── Inventory ──
    ax = axes_inv[col]
    ax.plot(t_s, m.inventory, color=color, lw=0.8, alpha=0.85)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_xlabel('Time (s)')
    if col == 0: ax.set_ylabel('Inventory (units)')

    # ── Inventory distribution ──
    ax = axes_hist[col]
    ax.hist(m.inventory, bins=40, color=color, edgecolor='none', alpha=0.8)
    ax.axvline(0, color='gray', lw=0.6, ls='--')
    ax.set_xlabel('Inventory (units)')
    if col == 0: ax.set_ylabel('Frequency')

# Row labels
for ax, label in zip([axes_pnl[0], axes_inv[0], axes_hist[0]],
                      ['Cumulative PnL', 'Inventory', 'Inventory Distribution']):
    ax.set_ylabel(f'{label}\n{ax.get_ylabel()}' if ax.get_ylabel() else label)

fig.suptitle('Market-Making Strategy Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: strategy_comparison.png')

## 5. Adverse selection analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (name, (bt, m)) in zip(axes, results.items()):
    color = COLORS[name]
    fills = bt._account.fills
    if not fills:
        ax.text(0.5, 0.5, 'No fills', ha='center', va='center', transform=ax.transAxes)
        continue

    # PnL change N=10 book-updates after each fill.
    ts_arr  = m.ts_us
    pnl_arr = m.pnl
    WINDOW  = 10
    adv_costs = []
    for fill in fills:
        idx = int(np.searchsorted(ts_arr, fill.ts_us))
        ahead = idx + WINDOW
        if ahead < len(pnl_arr):
            adv_costs.append(pnl_arr[ahead] - pnl_arr[idx])

    if adv_costs:
        ax.hist(adv_costs, bins=40, color=color, edgecolor='none', alpha=0.85)
        ax.axvline(np.mean(adv_costs), color='black', lw=1.5, ls='--',
                   label=f'Mean={np.mean(adv_costs):.4f}')
        ax.legend(fontsize=8)

    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('PnL change after fill ($)')
    if ax == axes[0]: ax.set_ylabel('Count')

fig.suptitle('Adverse Selection Cost Distribution (10-update window)', fontsize=12)
plt.tight_layout()
plt.savefig('adverse_selection.png', dpi=120)
plt.show()

## 6. Quote-to-trade latency (microseconds)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, (bt, m)) in zip(axes, results.items()):
    color = COLORS[name]
    lats  = bt.latency_us
    if not lats:
        ax.text(0.5, 0.5, 'No fills', ha='center', va='center', transform=ax.transAxes)
        continue
    ax.hist(lats, bins=50, color=color, edgecolor='none', alpha=0.85)
    ax.axvline(np.mean(lats), color='black', lw=1.5, ls='--',
               label=f'Mean={np.mean(lats):.1f} µs\nP99={np.percentile(lats,99):.1f} µs')
    ax.legend(fontsize=8)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Latency (µs)')
    if ax == axes[0]: ax.set_ylabel('Count')

fig.suptitle('Quote-to-Trade Latency Distribution', fontsize=12)
plt.tight_layout()
plt.savefig('latency.png', dpi=120)
plt.show()

## 7. Final metrics table (pretty)

In [ ]:
display(df.style
    .format(precision=4)
    .background_gradient(subset=['Total PnL', 'Sharpe'], cmap='RdYlGn')
    .background_gradient(subset=['Max Drawdown'], cmap='RdYlGn_r')
    .set_caption('Strategy Performance Summary')
)